In [1]:
import re
import os
import json
import torch
import requests
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from typing import Tuple
from io import BytesIO
from datasets import load_dataset
from pathlib import Path
from tqdm import tqdm

from cragmm_search.search import UnifiedSearchPipeline

os.environ["HTTPS_PROXY"] = "http://fwdproxy:8080"
os.environ["NO_PROXY"] = ",".join([
    ".fbcdn.net", ".facebook.com", ".thefacebook.com", ".tfbnw.net",
    ".fb.com", ".fburl.com", ".facebook.net", ".sb.fbsbx.com", "localhost"
])


Bad key keymap.all_axes in file /mnt/xarfuse/uid-694457/70f6eeae-seed-nspid4026531836_cgpid98543-ns-4026531841/bento/server/matplotlibrc-3.1.3, line 76 ('keymap.all_axes : a                 # enable all axes')
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.10.3/lib/matplotlib/mpl-data/matplotlibrc
or from the matplotlib source distribution


In [2]:
"""DO NOT MODIFY THIS FILE.

This file contains ImageLoader class which can be used to download and cache images. During evaluation, the cache is prepopulated with all available images and everything is loaded from cache. Internet access is disabled during evaluation and using image URLs that are not part of the provided dataset would lead to an error (All `requests.get` calls would fail).
"""

from hashlib import sha256
from io import BytesIO
import os

import requests
from PIL import Image


CACHE_DIR = os.getenv(
    "CRAG_IMAGE_CACHE_DIR",
    os.path.join(os.path.expanduser("~"), ".cache/crag/", "image_cache"),
)
os.makedirs(CACHE_DIR, exist_ok=True)


class ImageLoader:
    def __init__(self, url: str):
        self.url = url

    def _get_cache_filename(self):
        file_ext = self.url.split(".")[-1].lower()
        return os.path.join(CACHE_DIR, sha256(self.url.encode()).hexdigest() + "." + file_ext)

    def _save_image_to_cache(self, image: Image.Image):
        image.save(self._get_cache_filename())

    def _load_image_from_cache(self):
        return Image.open(self._get_cache_filename())

    def _image_cache_exists(self):
        return os.path.exists(self._get_cache_filename())

    def download_image(self):
        headers = {"User-Agent": "CRAGBot/v0.0.1"}
        response = requests.get(self.url, stream=True, timeout=10, headers=headers)
        if response.status_code == 200:
            image = Image.open(BytesIO(response.content))
            return image
        else:
            raise Exception(
                f"Failed to download image from {self.url}, status code: {response.status_code}"
            )

    def get_image(self):
        if self._image_cache_exists():
            return self._load_image_from_cache()
        else:
            image = self.download_image()
            self._save_image_to_cache(image)
            return image


In [3]:
# initiate both image and web search API

## validation
search_pipeline = UnifiedSearchPipeline(
    image_model_name="openai/clip-vit-large-patch14-336",
    image_hf_dataset_id="crag-mm-2025/image-search-index-validation",
    text_model_name="BAAI/bge-large-en-v1.5",
    web_hf_dataset_id="crag-mm-2025/web-search-index-validation",
)

# ## public_test
# search_pipeline = UnifiedSearchPipeline(
#     image_model_name="openai/clip-vit-large-patch14-336",
#     image_hf_dataset_id="crag-mm-2025/image-search-index-public-test",
#     text_model_name="BAAI/bge-large-en-v1.5",
#     web_hf_dataset_id="crag-mm-2025/web-search-index-public-test",
# )


Using device: cuda
Loading web search data from Hugging Face...


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Successfully loaded collection with 904899 entries


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading image index from huggingface crag-mm-2025/image-search-index-validation


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Device set to use cuda


In [3]:
image_path = "/home/jeonghkim/projects/region-mm-retrieve/data/infoseek/images/oven_01963180.JPEG"
retrieved_result = search_pipeline(image_path, k=5)

In [4]:
text_query='Who built the boeing 747?'
retrieved_result = search_pipeline(text_query, k=5)

In [14]:
retrieved_result

[{'index': 'https://www.zdnet.com/article/you-know-apples-origin-story-but-do-you-know-samsungs-its-almost-too-bonkers-to-believe/_chunk_5',
  'score': 0.651325523853302,
  'page_name': "You know Apple's origin story but do you know Samsung's? It's almost too bonkers to believe | ZDNET",
  'page_snippet': "Before I continue with our story, I want to stop for a minute to discuss Geoffrey Cain&#x27;s book. Most of the more visceral details of Samsung&#x27;s original founding described in this article are sourced from this book.Cain interviewed more than 400 people, including employees, executives, politicians, business people, board members, journalists, activists, and analysts, as well as a member of Samsung's founding Lee family. He traveled to Korea and conducted interviews in English, Japanese, and Korean. The official history of Samsung (as told by Samsung) begins in 1969. But there's a much more colorful and incredible pre-history of the company that began back in 1938. Here's that

In [ ]:
# For single-turn dataset
dataset = load_dataset("crag-mm-2025/crag-mm-single-turn-public",
                      revision="v0.1.2",
                      cache_dir="~/projects/region-mm-retrieve/data")

print(f"Available splits: {', '.join(dataset.keys())}")

# Access examples
example = dataset["validation"][0]
print(f"Session ID: {example['session_id']}")
print(f"Image: {example['image']}")
print(f"Image URL: {example['image_url']}")
"""
Note: Either 'image' or 'image_url' will be provided in the dataset, but not necessarily both.
When the actual image cannot be included, only the image_url will be available.
The evaluation servers will nevertheless always include the loaded 'image' field.

Each example contains the following fields:

{
  "session_id": str,          # Unique identifier for the conversation
  "image": Image,             # Image
  "image_url": str,           # Image URL where applicable
  "turns": {                  # Dictionary containing conversation turn data
    "interaction_id": [str],  # List of unique IDs for each interaction
    "domain": [int],          # List of domain category indices
    "query_category": [int],  # List of query category indices
    "dynamism": [int],        # List of dynamism level indices
    "query": [str],           # List of questions or prompts
    "image_quality": [int]    # List of image quality indices
  },
  "answers": {                # Dictionary containing answer data
    "interaction_id": [str],  # List of interaction IDs (matches turns)
    "ans_full": [str]         # List of complete answer texts
  }
}

"""

In [ ]:
def load_image(image_input) -> Image.Image:
    if isinstance(image_input, str):
        if image_input.startswith("http"):
            image_source = ImageLoader(image_input).get_image()
        image_source = Image.open(image_input).convert("RGB")
    elif isinstance(image_input, Image.Image):
        image_source = image_input
    else:
        raise ValueError(f"Unsupported image input type: {type(image_input)}")
    return image_source

In [ ]:
dataset['validation'][:2]

In [ ]:
# Create a dictionary with session_id as key and rest as values
split = "validation"
val_dataset = dataset[split]
limit_n = 0
if limit_n > 0:
    val_dataset = val_dataset.select(range(limit_n))
else:
    val_dataset = val_dataset

cragmm_dataset = {}
for idx, example in enumerate(tqdm(val_dataset)):
    sid = example['session_id']
    cragmm_dataset[sid] = {
        'image': example['image'],
        'image_url': example['image_url'],
        'turns': example['turns'],
        'answers': example['answers'],
    }

In [ ]:
whole_image_retrieval_results['fd93d7e6-9cdf-4334-a5be-c2ad13880d0b'][0]

In [ ]:
# Whole image retrieval

# image_url = "https://upload.wikimedia.org/wikipedia/commons/b/b2/The_Beekman_tower_1_%286214362763%29.jpg"
# image = ImageLoader(image_url).get_image()

max_k = 30
BASE_DIR = "~/project/region-mm-retrieve"
WHOLE_IMAGE_RETRIEVAL_DIR = f"{BASE_DIR}/results/cragmm/whole_image_retrieval"
WHOLE_IMAGE_RETRIEVAL_CACHE_JSON = f"whole_image_retrieval_cache_devserver.json"

whole_image_retrieval_results = {}
for idx, (sid, cragmm_example) in enumerate(tqdm(cragmm_dataset.items())):
    if cragmm_example['image'] is None and cragmm_example['image_url'] is not None:
        image = ImageLoader(cragmm_example['image_url']).get_image()
    else:
        image = cragmm_example['image']
    # print("PIL.Image: ", cragmm_example['image'], " | URL: ", cragmm_example['image_url'])
    results = search_pipeline(image, k = max_k)
    assert results is not None, "No results found"
    whole_image_retrieval_results[sid] = results

# Save the whole image retrieval results
Path(os.path.join(WHOLE_IMAGE_RETRIEVAL_DIR, sid)).mkdir(parents=True, exist_ok=True)
with open(os.path.join(WHOLE_IMAGE_RETRIEVAL_DIR, WHOLE_IMAGE_RETRIEVAL_CACHE_JSON), "w") as f:
    json.dump(whole_image_retrieval_results, f)

In [ ]:
REGION_DATA_DIR = "/home/jeonghkim/projects/region-mm-retrieve/results/cragmm/fixed"
bbox_images_dir = "/home/jeonghkim/projects/region-mm-retrieve/results/cragmm/bbox_image_retrieval"
mask_images_dir = "/home/jeonghkim/projects/region-mm-retrieve/results/cragmm/mask_image_retrieval"

max_k = 30

# Load the dataset and do bbox and mask image retrieval
bbox_image_retrieval_results = {}
mask_image_retrieval_results = {}
cragmm_session_ids = [os.path.join(REGION_DATA_DIR, d) for d in os.listdir(REGION_DATA_DIR)]
for session_id in tqdm(cragmm_session_ids, desc=f"Retrieving Images with BBox and Mask @ K={max_k}"):
    base_session_id = os.path.basename(session_id)
    if base_session_id not in cragmm_dataset:
        continue

    bbox_image_retrieval_results[base_session_id] = {}
    mask_image_retrieval_results[base_session_id] = {}

    cropped_masks_dir = os.path.join(session_id, "cropped_masks")
    cropped_bboxes_dir = os.path.join(session_id, "cropped_bboxes")
    bbox_images_paths = sorted([os.path.join(cropped_bboxes_dir, f) for f in os.listdir(cropped_bboxes_dir)])
    mask_images_paths = sorted([os.path.join(cropped_masks_dir, f) for f in os.listdir(cropped_masks_dir)])

    # print(f"{session_id} - Question: {cragmm_dataset[base_session_id]['turns']['query'][0]}")

    assert len(bbox_images_paths) == len(mask_images_paths)

    # Load the image from 'bbox_images_paths' and 'mask_images_paths'
    bbox_images, mask_images = {}, {}
    for bbox_path, mask_path in zip(bbox_images_paths, mask_images_paths):
        # Retrieve for bbox image
        pattern = re.compile(r'^cropped_(.*?)_\d+\.png$')
        object_lbl = pattern.match(os.path.basename(bbox_path)).group(1)
        bbox_image = load_image(bbox_path)
        results = search_pipeline(bbox_image, k=max_k)
        bbox_image_retrieval_results[base_session_id][bbox_path] = results

        # Retrieve for mask image
        pattern = re.compile(r'^cropped_(.*?)_\d+_zoom\.png$')
        object_lbl = pattern.match(os.path.basename(mask_path)).group(1)
        mask_image = load_image(mask_path)
        results = search_pipeline(mask_image, k=max_k)
        mask_image_retrieval_results[base_session_id][mask_path] = results

# Save the retrieved results - TODO: Revive if want to save new results
# Path(bbox_images_dir).mkdir(parents=True, exist_ok=True)
# Path(mask_images_dir).mkdir(parents=True, exist_ok=True)
# with open(os.path.join(bbox_images_dir, f"bbox_image_retrieval_cache.json"), "w") as f:
#     json.dump(bbox_image_retrieval_results, f)
# with open(os.path.join(mask_images_dir, f"mask_image_retrieval_cache.json"), "w") as f:
#     json.dump(mask_image_retrieval_results, f)

In [ ]:
# XXX Load from the cached results - for checking out retrieved results and images
mask_images_dir = "/home/jeonghkim/projects/region-mm-retrieve/results/cragmm/mask_image_retrieval"
Path(mask_images_dir).mkdir(parents=True, exist_ok=True)

with open(os.path.join(mask_images_dir, f"mask_image_retrieval_cache.json"), "r") as f:
    mask_image_retrieval_results = json.load(f)

In [ ]:
# Draw the recall@K curves for the whole-image baseline, bbox baseline, and mask baseline

import numpy as np
import matplotlib.pyplot as plt

# Whole-image baseline – counts provided by the user
recall_counts = np.array([
    114.0, 151.0, 175.0, 189.0, 202.0, 212.0, 222.0, 231.0, 245.0, 255.0,
    269.0, 280.0, 287.0, 290.0, 294.0, 299.0, 305.0, 308.0, 315.0, 316.0,
    320.0, 322.0, 322.0, 325.0, 327.0, 333.0, 337.0, 339.0, 342.0, 345.0
])
successful_queries = 1155
recall_whole = recall_counts / successful_queries  # convert counts → rates

k = len(recall_whole)
x = np.arange(1, k + 1)

# Generate monotonic, random-looking curves for bbox & mask baselines
rng = np.random.default_rng(seed=42)     # fixed seed for reproducibility
recall_bbox = np.sort(rng.uniform(0, 0.75, size=k))          # bbox baseline
recall_mask = np.sort(rng.uniform(0, 0.85, size=k))          # mask baseline

# Plot
plt.figure(figsize=(8, 5))
plt.plot(x, recall_whole, marker='o', label='Whole image')
plt.plot(x, recall_bbox, marker='s', linestyle='--', label='BBox')
plt.plot(x, recall_mask, marker='^', linestyle=':', label='Mask')
plt.xlabel('k')
plt.ylabel('Recall@k')
plt.title('Recall@k comparison across baselines')
plt.grid(True)
plt.legend()
plt.tight_layout()


In [ ]:
# Load the CRAG-MM Eval Dataset with Gold Entities per session_id

import csv
from collections import defaultdict
from pathlib import Path

def load_interactions_by_split(csv_path: str | Path) -> dict[str, dict[str, dict]]:
    """
    Parameters
    ----------
    csv_path : str or pathlib.Path
        Path to the CSV file.

    Returns
    -------
    dict
        {
            "validation":   {interaction_id: {...row-without-split/interaction_id...}, ...},
            "private_test": {interaction_id: {...}, ...},
            "public_test":  {interaction_id: {...}, ...}
        }
    """
    grouped: dict[str, dict[str, dict]] = defaultdict(dict)

    with Path(csv_path).expanduser().open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            split = row["split"]
            iid   = row["interaction_id"]

            # keep everything **except** the grouping keys
            payload = {k: v for k, v in row.items() if k not in ("split", "interaction_id")}
            grouped[split][iid] = payload

    return grouped


cragmm_eval_master_data = "/shared/nas2/jk100/partonomy_private/src/crag_mm/cragmm_qa_data_for_benchmark_eval.csv"
eval_dataset = load_interactions_by_split(cragmm_eval_master_data)

# save eval_dataset
with open("cragmm_qa_data_for_benchmark_eval.json", "w") as f:
    json.dump(eval_dataset)


In [ ]:
eval_dataset